In [1]:
!pip install -q transformers datasets torch pandas numpy accelerate

In [2]:
from google.colab import files

print("Upload solution_C_model.zip and one input CSV file")
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

Upload solution_C_model.zip and one input CSV file


Saving solution_C_model.zip to solution_C_model.zip
Saving test.csv to test.csv
Uploaded files: ['solution_C_model.zip', 'test.csv']


In [3]:
import zipfile
import os
import shutil

zip_name = "solution_C_model.zip"
extract_dir = "solution_C_model"

if not os.path.exists(zip_name):
    raise FileNotFoundError(f"Could not find {zip_name}. Make sure you uploaded it.")

if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted model to:", extract_dir)
print("Files inside extracted folder:")
print(os.listdir(extract_dir))

Extracted model to: solution_C_model
Files inside extracted folder:
['config.json', 'tokenizer_config.json', 'model.safetensors', 'tokenizer.json']


## Demo Code – Solution C

This notebook loads the saved transformer-based NLI model and generates predictions for an input CSV file containing premise-hypothesis pairs.

In [4]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

csv_files = [f for f in uploaded.keys() if f.endswith(".csv")]

if len(csv_files) != 1:
    raise ValueError(f"Expected exactly one input CSV, found: {csv_files}")

input_csv = csv_files[0]
input_df = pd.read_csv(input_csv)

required_cols = {"premise", "hypothesis"}
if not required_cols.issubset(input_df.columns):
    raise ValueError(f"Input CSV must contain columns {required_cols}, found {input_df.columns.tolist()}")

tokenizer = AutoTokenizer.from_pretrained("solution_C_model")
model = AutoModelForSequenceClassification.from_pretrained("solution_C_model")

print("Loaded input file:", input_csv)
print("Input shape:", input_df.shape)
display(input_df.head())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loaded input file: test.csv
Input shape: (3302, 2)


,premise,hypothesis
0,"Boy wearing red hat, blue jacket pushing plow ...",The boy is surrounded by snow
1,A blond woman in a black shirt is standing beh...,The woman is standing.
2,Three people in uniform are outdoors and are o...,Uniformed people are outside
3,"A person, in a striped blue shirt and pants, i...",The person is running
4,"A man, woman, and child get their picture take...",A family on vacation is posing.


In [5]:
from datasets import Dataset

input_ds = Dataset.from_pandas(
    input_df[["premise", "hypothesis"]],
    preserve_index=False
)

print(input_ds)

Dataset({
    features: ['premise', 'hypothesis'],
    num_rows: 3302
})


In [6]:
MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

input_tok = input_ds.map(tokenize_batch, batched=True)
input_tok = input_tok.remove_columns(["premise", "hypothesis"])
input_tok.set_format("torch")

print(input_tok)

Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3302
})


In [7]:
import numpy as np
from transformers import Trainer

trainer = Trainer(model=model)

output = trainer.predict(input_tok)
pred = np.argmax(output.predictions, axis=-1)

print("Generated predictions:", len(pred))
print(pd.Series(pred).value_counts())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Generated predictions: 3302
1    1719
0    1583
Name: count, dtype: int64


In [8]:
GROUP_NUMBER = "56"

pred_df = pd.DataFrame({"prediction": pred})
output_name = f"Group_{GROUP_NUMBER}_C.csv"
pred_df.to_csv(output_name, index=False)

print(f"Saved {output_name}")
display(pred_df.head())

Saved Group_56_C.csv


,prediction
0,1
1,1
2,1
3,1
4,1


In [9]:
files.download(output_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>